In [1]:
import langchain



In [2]:
###introduction to data ingestion and parsing
import os
from typing import List,Dict,Any
import pandas as pd



In [23]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

prompt = ChatPromptTemplate.from_template("""You are a university student assistant.
Answer the question using only the context below.

Rules:
1. Use only the context. Do not use outside knowledge.
2. If the context does not contain the answer, say: "The provided documents don't say."
3. Never guess numbers, dates or rules.
4. Keep the answer short and simple.

Context:
{context}

Question: {question}

Answer:""")

THRESHOLD = 0.75

def search_or_refuse(question, k=3):
    results = vectorstore.similarity_search_with_score(question, k=k)
    return [doc for doc, score in results if score <= THRESHOLD]

def ask(question):
    docs = search_or_refuse(question)
    if not docs:
        return "I couldn't find this in the university documents.", []
    context = "\n\n".join(d.page_content for d in docs)
    raw_answer = llm.invoke(prompt.format_messages(context=context, question=question)).content
    if isinstance(raw_answer, list):
        answer = "".join(b["text"] for b in raw_answer if isinstance(b, dict) and b.get("type") == "text")
    else:
        answer = raw_answer
    return answer, sorted({d.metadata["source"] for d in docs})

print("Setup complete")

Setup complete


In [3]:
import langchain_core
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)

print("setup completed")


c:\rag udemy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup completed


### understanding data ingestion and parsing


In [24]:
#create a document
doc=document(
    page_content="This is a sample document for testing the data ingestion and parsing functionality.",
    metadata={"source": "test_document", "author": "John Doe", "psg": "sample_paragraph"
    "page_number": 1, "date": "2023-10-01","custom_field": "any_value"}
)
print("Document structure")

print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

SyntaxError: invalid syntax (1729158316.py, line 5)

### text files-(simple case)

In [ ]:
## create a simple text file
import os
os.makedirs("data/text_files", exist_ok=True)

In [ ]:
sample_texts= {
    "data/text_files/python_intro.txt": """
    The Future of Renewable Energy

Renewable energy sources have become central to global efforts to combat climate change and reduce dependence on fossil fuels. Solar, wind, hydroelectric, and geothermal power now account for a growing share of electricity generation worldwide.

Solar Power
Solar energy is harvested using photovoltaic cells that convert sunlight directly into electricity. Over the past decade, the cost of solar panels has dropped dramatically, making solar one of the cheapest sources of new electricity generation in many parts of the world. Countries like India and China have made massive investments in solar infrastructure, with India's National Solar Mission aiming to significantly expand solar capacity in the coming years.

Wind Energy
Wind turbines convert kinetic energy from wind into electrical power. Modern wind farms, both onshore and offshore, can generate substantial amounts of clean energy. Offshore wind farms, in particular, benefit from stronger and more consistent wind patterns, though they come with higher installation and maintenance costs.

Hydroelectric Power
Hydroelectric power remains one of the oldest and most reliable forms of renewable energy. By using flowing water to spin turbines, hydroelectric plants can provide a stable base load of electricity. However, large dam projects can have significant environmental and social impacts, including displacement of communities and disruption of river ecosystems.

Geothermal Energy
Geothermal power taps into the Earth's internal heat to generate electricity or provide direct heating. While geothermal energy is highly reliable and has a small land footprint, it is geographically limited to regions with accessible geothermal activity, such as Iceland and parts of the United States.

Challenges and the Road Ahead
Despite rapid growth, renewable energy still faces challenges including energy storage limitations, grid integration issues, and the intermittent nature of solar and wind power. Advances in battery technology, smart grids, and energy policy will play a crucial role in accelerating the global transition to a fully renewable energy future. 
"""""
}

for file_path, content in sample_texts.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)


print("sample text files created:")

sample text files created:


In [11]:
new_files = {
    "data/text_files/attendance_policy.txt": """Attendance Policy

Students must maintain a minimum of 75 percent attendance in each subject
to be eligible for end-semester examinations. Students with attendance
between 65 and 75 percent may apply for condonation with a valid medical
certificate. Attendance below 65 percent will result in a detained status
for that subject.""",
    "data/text_files/exam_rules.txt": """the timing for the exam is from 7 AM to 10 AM,no bags or any electronic device is not allowed during the exam and any student is found with some cheating material will be provided zero marks.""",
    "data/text_files/library_rules.txt": """the timings of the session is from 8 AM to 8 PM ,No noise should be created in the library""",
}

for file_path, content in new_files.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

In [12]:
print(f"Loaded {len(documents)} documents")
for d in documents:
    print(d.metadata["source"])

NameError: name 'documents' is not defined

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:200])
print(docs[0].metadata)

C:\Users\avish\AppData\Local\Temp\ipykernel_24896\280556110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\rag udemy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = dir_loader.load()

print(f"Loaded {len(documents)} documents")
for d in documents:
    print(d.metadata["source"], "->", len(d.page_content), "characters")

Loaded 7 documents
data\text_files\antiragging_policy.txt -> 2983 characters
data\text_files\attendance_policy.txt -> 1645 characters
data\text_files\exam_rules.txt -> 3180 characters
data\text_files\fees_structure.txt -> 1907 characters
data\text_files\library_rules.txt -> 1935 characters
data\text_files\python_intro.txt -> 2166 characters
data\text_files\uniform_rules.txt -> 1494 characters


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = splitter.split_documents(documents)

print(f"{len(documents)} documents became {len(chunks)} chunks")

7 documents became 44 chunks


In [6]:
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print(chunk.metadata)
    print()

--- Chunk 0 (486 chars) ---
6. Anti-Ragging Policy
6.1 Purpose

The institution prohibits ragging and is committed to providing a safe, respectful, and inclusive learning environment for all students.

Ragging includes conduct that causes, or is likely to cause, physical or psychological harm, humiliation, fear, or distress to a student, particularly a fresher. It may occur in person, online, in hostels, or during college-related activities.

6.2 Prohibited Activities

The following activities are prohibited:
{'source': 'data\\text_files\\antiragging_policy.txt'}

--- Chunk 1 (451 chars) ---
Forcing a student to perform an embarrassing, humiliating, or degrading act.
Using abusive language, threats, intimidation, or discriminatory remarks.
Forcing juniors to introduce themselves in an insulting or degrading manner.
Physically assaulting, harassing, or threatening another student.
Forcing students to participate in unwanted activities or social events.
Demanding money, belongings, servi

In [ ]:
import sys
print(sys.executable)


c:\rag udemy\.venv\Scripts\python.exe


In [ ]:
%pip install faiss-cpu langchain-google-genai python-dotenv --default-timeout=100 --retries 10


Note: you may need to restart the kernel to use updated packages.


c:\rag udemy\.venv\Scripts\python.exe: No module named pip


In [34]:
import faiss
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv

print("all imports OK")

all imports OK


In [35]:
print(f"Loaded {len(documents)} documents")
for d in documents:
    print(d.metadata["source"])
print(f"{len(chunks)} chunks")

Loaded 7 documents
data\text_files\antiragging_policy.txt
data\text_files\attendance_policy.txt
data\text_files\exam_rules.txt
data\text_files\fees_structure.txt
data\text_files\library_rules.txt
data\text_files\python_intro.txt
data\text_files\uniform_rules.txt
44 chunks


In [9]:
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vectors stored:", vectorstore.index.ntotal)

vectorstore.save_local("faiss_index")

Vectors stored: 44


In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
print("Key found:", os.getenv("GOOGLE_API_KEY") is not None)

Key found: True


In [36]:
print("Chunks:", len(chunks))
print("Vectors stored:", vectorstore.index.ntotal)

Chunks: 44
Vectors stored: 44


In [39]:
results = vectorstore.similarity_search("How many classes must I attend?", k=6)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} | {doc.metadata['source']} ---")
    print(doc.page_content)
    print()

--- Result 1 | data\text_files\attendance_policy.txt ---
1.2 Attendance Calculation

Attendance is calculated based on the number of classes attended divided by the total number of classes conducted, multiplied by 100.

Attendance Percentage = (Number of Classes Attended / Total Classes Conducted) × 100

For example, if a student attends 36 out of 45 classes, their attendance is 80%.

1.3 Shortage of Attendance

--- Result 2 | data\text_files\attendance_policy.txt ---
1.4 Medical Leave

Students who are absent due to illness or medical emergencies should inform their class coordinator and submit supporting medical documents as required. Submission of medical documents does not automatically guarantee that attendance will be granted or that a shortage will be condoned.

1.5 Attendance Monitoring

--- Result 3 | data\text_files\attendance_policy.txt ---
1. Attendance Policy
1.1 Minimum Attendance Requirement

All students are expected to attend lectures, practical sessions, tutorials, an

In [38]:
results = vectorstore.similarity_search_with_score("What is the capital of France", k=6)

for doc, score in results:
    print(round(score, 3), "|", doc.metadata["source"])

0.904 | data\text_files\python_intro.txt
0.918 | data\text_files\python_intro.txt
0.924 | data\text_files\python_intro.txt
0.924 | data\text_files\python_intro.txt
0.93 | data\text_files\fees_structure.txt
0.931 | data\text_files\python_intro.txt


In [40]:
queries = [
    "How many classes must I attend?",
    "What is the fine for returning a book late?",
    "What is the capital of France?",
    "Who won the cricket world cup?"
    "what is the timing for the exam?"
    "what is anti ragging and how to prevent it?",
]

for q in queries:
    res = vectorstore.similarity_search_with_score(q, k=6)
    print(q)
    for doc, score in res:
        print("   ", round(score, 3), "|", doc.metadata["source"])

How many classes must I attend?
    0.544 | data\text_files\attendance_policy.txt
    0.572 | data\text_files\attendance_policy.txt
    0.586 | data\text_files\attendance_policy.txt
    0.593 | data\text_files\attendance_policy.txt
    0.618 | data\text_files\attendance_policy.txt
    0.715 | data\text_files\exam_rules.txt
What is the fine for returning a book late?
    0.49 | data\text_files\library_rules.txt
    0.601 | data\text_files\library_rules.txt
    0.651 | data\text_files\library_rules.txt
    0.722 | data\text_files\library_rules.txt
    0.747 | data\text_files\library_rules.txt
    0.757 | data\text_files\fees_structure.txt
What is the capital of France?
    0.889 | data\text_files\python_intro.txt
    0.902 | data\text_files\python_intro.txt
    0.908 | data\text_files\python_intro.txt
    0.914 | data\text_files\python_intro.txt
    0.915 | data\text_files\fees_structure.txt
    0.918 | data\text_files\python_intro.txt
Who won the cricket world cup?what is the timing for

In [42]:
borderline = [
    "What is the last date to pay the semester fee?",
    "Is there a hostel curfew?",
    "Can I take a book home during the holidays?",
    "What are the exam timings?",
]

for q in borderline:
    res = vectorstore.similarity_search_with_score(q, k=6)
    print(q)
    for doc, score in res:
        print("   ", round(score, 3), "|", doc.metadata["source"])

What is the last date to pay the semester fee?
    0.566 | data\text_files\fees_structure.txt
    0.61 | data\text_files\fees_structure.txt
    0.623 | data\text_files\fees_structure.txt
    0.647 | data\text_files\fees_structure.txt
    0.682 | data\text_files\exam_rules.txt
    0.701 | data\text_files\fees_structure.txt
Is there a hostel curfew?
    0.746 | data\text_files\library_rules.txt
    0.752 | data\text_files\library_rules.txt
    0.764 | data\text_files\antiragging_policy.txt
    0.793 | data\text_files\library_rules.txt
    0.793 | data\text_files\antiragging_policy.txt
    0.794 | data\text_files\library_rules.txt
Can I take a book home during the holidays?
    0.616 | data\text_files\library_rules.txt
    0.62 | data\text_files\library_rules.txt
    0.641 | data\text_files\library_rules.txt
    0.652 | data\text_files\library_rules.txt
    0.684 | data\text_files\library_rules.txt
    0.702 | data\text_files\library_rules.txt
What are the exam timings?
    0.623 | data\t

In [43]:
THRESHOLD = 0.85  # your choice, with your reasoning

def search_or_refuse(question, k=6):
    results = vectorstore.similarity_search_with_score(question, k=k)
    good = [doc for doc, score in results if score <= THRESHOLD]
    return good   # an empty list means "not found in the documents"

In [44]:
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = FAISS.load_local(
    "faiss_index", embeddings, allow_dangerous_deserialization=True
)
print("Vectors stored:", vectorstore.index.ntotal)   # expect 9

Vectors stored: 44


In [26]:
THRESHOLD = 0.85

def search_or_refuse(question, k=6):
    results = vectorstore.similarity_search_with_score(question, k=k)
    good = [doc for doc, score in results if score <= THRESHOLD]
    return good

In [45]:
THRESHOLD = 0.85

tests = {
    "How many classes must I attend?": True,
    "What is the fine for returning a book late?": True,
    "Can I take a book home during the holidays?": True,
    "What are the exam timings?": True,
    "Is there a hostel curfew?": False,
    "What is the last date to pay the semester fee?": False,
    "What is the capital of France?": False,
    "Who won the cricket world cup?": False,
}

for q, should_answer in tests.items():
    good = search_or_refuse(q)
    answered = len(good) > 0
    flag = "OK " if answered == should_answer else "BUG"
    print(flag, "| kept", len(good), "chunk(s) |", q)

OK  | kept 6 chunk(s) | How many classes must I attend?
OK  | kept 6 chunk(s) | What is the fine for returning a book late?
OK  | kept 6 chunk(s) | Can I take a book home during the holidays?
OK  | kept 6 chunk(s) | What are the exam timings?
BUG | kept 6 chunk(s) | Is there a hostel curfew?
BUG | kept 6 chunk(s) | What is the last date to pay the semester fee?
OK  | kept 0 chunk(s) | What is the capital of France?
OK  | kept 0 chunk(s) | Who won the cricket world cup?


In [28]:
import os

for name in os.listdir("data/text_files"):
    path = os.path.join("data/text_files", name)
    with open(path, encoding="utf-8") as f:
        text = f.read()
    print(name, "|", len(text), "chars |", repr(text[:60]))

antiragging_policy.txt | 2983 chars | '6. Anti-Ragging Policy\n6.1 Purpose\n\nThe institution prohibit'
attendance_policy.txt | 1645 chars | '1. Attendance Policy\n1.1 Minimum Attendance Requirement\n\nAll'
exam_rules.txt | 3180 chars | '7. Examination Rules and Regulations\n7.1 Examination Eligibi'
fees_structure.txt | 1907 chars | '. College Fees and Payment Rules\n4.1 Fee Payment\n\nAll studen'
library_rules.txt | 1935 chars | '2. Library Rules and Regulations\n2.1 Library Membership\n\nAll'
python_intro.txt | 2166 chars | '\n    The Future of Renewable Energy\n\nRenewable energy source'
uniform_rules.txt | 1494 chars | 'College Uniform and Dress Code\n5.1 Uniform Requirement\n\nStud'


In [46]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [48]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

docs = dir_loader.load()
chunks = splitter.split_documents(docs)
print(len(docs), "files ->", len(chunks), "chunks")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db = FAISS.from_documents(chunks, embeddings)
db.save_local("faiss_index")

7 files -> 44 chunks


In [47]:
documents = dir_loader.load()
chunks = splitter.split_documents(documents)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print("Chunks:", len(chunks), "| Vectors:", vectorstore.index.ntotal)

Chunks: 44 | Vectors: 44


In [31]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful library assistant. Answer the user's question
using only the information provided in the context.

Instructions:
1. Read the context carefully before answering.
2. Give a clear, accurate, and concise answer based only on the context.
3. Do not make up information or use outside knowledge.
4. If the context does not contain the answer, politely say
   that the information is not available in the provided documents.
5. If the question is about a library fine, mention the exact
   amount and any relevant conditions, if available.
6. Use simple and easy-to-understand language.

Context:
{context}

Question: {question}

Answer:
""")

chunks_found = search_or_refuse(
    "What is the fine for returning a book late?"
)

context = "\n\n".join(c.page_content for c in chunks_found)

print(prompt.format(
    context=context,
    question="What is the fine for returning a book late?"
))

Human: 
You are a helpful library assistant. Answer the user's question
using only the information provided in the context.

Instructions:
1. Read the context carefully before answering.
2. Give a clear, accurate, and concise answer based only on the context.
3. Do not make up information or use outside knowledge.
4. If the context does not contain the answer, politely say
   that the information is not available in the provided documents.
5. If the question is about a library fine, mention the exact
   amount and any relevant conditions, if available.
6. Use simple and easy-to-understand language.

Context:
Students are responsible for returning books on or before the due date. Late returns may attract a fine according to the library's prescribed fine schedule.

Sample rule for RAG testing: A late fine of ₹5 per book per day may be charged for overdue books. The actual fine and any maximum limit must be confirmed from the official library policy.

2.5 Lost or Damaged Books

2.3 Book B

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

prompt = ChatPromptTemplate.from_template("""You are a university student assistant.
Answer the question using only the context below.

Rules:
1. Use only the context. Do not use outside knowledge.
2. If the context does not contain the answer, say: "The provided documents don't say."
3. Never guess numbers, dates or rules.
4. Keep the answer short and simple.

Context:
{context}

Question: {question}

Answer:""")

def ask(question):
    docs = search_or_refuse(question)
    if not docs:
        return "I couldn't find this in the university documents.", []
    context = "\n\n".join(d.page_content for d in docs)
    answer = llm.invoke(prompt.format_messages(context=context, question=question)).content
    return answer, sorted({d.metadata["source"] for d in docs})

tests = [
    "What is the minimum attendance required?",     # answerable
    "What are the exam timings?",                   # answerable (7 AM to 10 AM)
    "What are the library timings?",                # answerable (8 AM to 8 PM)
    "What is the fine for returning a book late?",  # passes search, fact missing
    "Is there a hostel curfew?",                    # refused by threshold
    "What is the capital of France?",               # refused by threshold
]

for q in tests:
    answer, sources = ask(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:", sources)
    print()

GoogleModelNotFoundError: Error calling model 'gemini-2.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:
from google import genai

client = genai.Client()   # reads GOOGLE_API_KEY from your environment

for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

prompt = ChatPromptTemplate.from_template("""You are a university student assistant.
Answer the question using only the context below.

Rules:
1. Use only the context. Do not use outside knowledge.
2. If the context does not contain the answer, say: "The provided documents don't say."
3. Never guess numbers, dates or rules.
4. Keep the answer short and simple.

Context:
{context}

Question: {question}

Answer:""")

def ask(question):
    docs = search_or_refuse(question)
    if not docs:
        return "I couldn't find this in the university documents.", []
    context = "\n\n".join(d.page_content for d in docs)
    answer = llm.invoke(prompt.format_messages(context=context, question=question)).content
    return answer, sorted({d.metadata["source"] for d in docs})

tests = [
    "What is the minimum attendance required?",     # answerable
    "What are the exam timings?",                   # answerable (7 AM to 10 AM)
    "What are the library timings?",                # answerable (8 AM to 8 PM)
    "What is the fine for returning a book late?",  # passes search, fact missing
    "Is there a hostel curfew?",                    # refused by threshold
    "What is the capital of France?",               # refused by threshold
]

for q in tests:
    answer, sources = ask(q)
    print("Q:", q)
    print("A:", answer)
    print("Sources:", sources)
    print()

Q: What is the minimum attendance required?
A: [{'type': 'text', 'text': 'Students must maintain at least 75% attendance in each subject to be eligible to appear in semester examinations, subject to applicable university regulations.', 'extras': {'signature': 'EnMKcQFpFH0TdF6meyLInA6Q39Zr0cj6X1Rt1J1xZLF3g3kRaoIqjg02OI+qo1hAGQTUy/JehIyyndcAe8KJe50TlA53QNcvqtjiYFYomE9LujuXoa+e7fBHZgu6siwo8QLDU5T+KpVVWsdU2pzllhS4r9Nb'}}]
Sources: ['data\\text_files\\attendance_policy.txt', 'data\\text_files\\exam_rules.txt']

Q: What are the exam timings?
A: [{'type': 'text', 'text': "The provided documents don't say.", 'extras': {'signature': 'EnMKcQFpFH0TQHlZH7t6pqLNCboPpbbtX0R69ZxNTJAwGiTi4+xdgypfzYrvycZyQ7g+gET/ZJc3zRZlTS+fj1Gr6YwK93/s2qUjZzOHMD1lqXNilcbMuMxCP/jqpxItL6/dwLg7uJpC1Gyu9kL6ZCbMHU96'}}]
Sources: ['data\\text_files\\exam_rules.txt']

Q: What are the library timings?
A: [{'type': 'text', 'text': 'The library operates during the working hours notified by the institution. Timings may vary duri

In [22]:
docs = search_or_refuse("What is the fine for returning a book late?")
for d in docs:
    print(d.metadata["source"], "->", repr(d.page_content))

data\text_files\library_rules.txt -> "Students are responsible for returning books on or before the due date. Late returns may attract a fine according to the library's prescribed fine schedule.\n\nSample rule for RAG testing: A late fine of ₹5 per book per day may be charged for overdue books. The actual fine and any maximum limit must be confirmed from the official library policy.\n\n2.5 Lost or Damaged Books"
data\text_files\library_rules.txt -> "2.3 Book Borrowing Rules\nStudents must present their valid college ID card when borrowing books.\nBooks may be issued only to registered library members.\nThe number of books permitted and the loan period are determined by the library's current borrowing policy.\nReference books, journals, newspapers, and other restricted materials may only be used within the library.\nBorrowed books must be returned or renewed before their due date, subject to library rules.\n2.4 Late Return and Fines"
data\text_files\library_rules.txt -> '2.5 Lost or Dam

In [49]:
for d in db.similarity_search("what is the fees structure", k=4):
    print(d.metadata.get("source"), "|", d.page_content[:150].replace("\n", " "))
    print("---")

data\text_files\fees_structure.txt | Tuition fees Registration or admission fees Examination fees Library fees Laboratory fees Hostel and mess fees, where applicable Security deposits Oth
---
data\text_files\fees_structure.txt | . College Fees and Payment Rules 4.1 Fee Payment  All students must pay tuition fees and other applicable institutional charges according to the fee s
---
data\text_files\fees_structure.txt | Late fees, if applicable, will be charged according to the official fee circular. Students should contact the accounts office if they encounter paymen
---
data\text_files\fees_structure.txt | 4.6 Outstanding Dues  Students with unpaid fees may be subject to restrictions on services or administrative processes where permitted by institutiona
---
